# 🤖 Tinh Chỉnh Ngắn PPO 9D + Phạt Vận Tốc Quanh Gốc (`15 PWM`, `5 cm / 2° / 3 cm/s / max 0.02`, 50k bước, `lr = 3e-5`)

**Cấu hình đồng bộ với Firmware `Project_self` (STM32L476RG):**
- **Chu kỳ điều khiển:** 200 Hz (`Ts = 5 ms`), Encoder cập nhật mỗi `20 ms` (50 Hz, 1400 CPR, bộ lọc vận tốc $N=10$).
- **Chuỗi truyền động giữ nguyên mốc:** Hệ số khuếch đại PPO `1.30x` → Bộ lọc PWM `0.60 × u_raw + 0.40 × u_prev` → Làm tròn số nguyên → Bù vùng chết mềm `15 PWM` (`|u| ≤ 2 → 0`, `deadband = 80 PWM`).
- **Thay đổi duy nhất trong `src/environment.py`:** Phạt thêm vận tốc liên tục khi gần thẳng và gần gốc: $0.02 \cdot \exp(-(x/0.05)^2) \cdot \exp(-(\theta/2^\circ)^2) \cdot \tanh(|v|/0.03)$ (giữ nguyên `w_pos = 0.30`, `w_action_diff = 1.60`, `r_hold_bonus = 0.0`).
- **Khởi tạo ấm & đánh giá mở rộng:**
  - Nạp đủ trọng số & bias 3 tầng actor từ `best_policy_wpos_030_sym.npz` (`57/98` giữ VT, `24/98` ngã), train `50.000` bước (`lr = 3e-5`, chấm mỗi `10.000` bước).
  - Giữ bước 0 trong bảng so sánh; chấm đủ **30 ca giữ VT**, **20 ca đẩy `7000–7009`**, **20 ca seed mới `7100–7109`** và **98 ca** (`real_robot` 10s).

---
### Bước 1: Lấy code mới nhất từ GitHub, cài đặt thư viện & định nghĩa `run_step` (dừng ngay khi lỗi)

> **Lưu ý:** `git reset --hard` giữ nguyên các checkpoint cũ trong `outputs/checkpoints/` nếu bạn đang chạy tiếp trên cùng phiên Colab.

In [ ]:
import os
import sys
import subprocess

def run_step(cmd, required_files=None):
    """Chạy lệnh đồng bộ và DỪNG NGAY nếu lệnh trả về lỗi hoặc thiếu file đầu ra bắt buộc."""
    print(f"\n>>> RUN: {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd)
    if proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại với mã lỗi {proc.returncode}: {' '.join(cmd)}")
    if required_files:
        missing = [f for f in required_files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(f"Thiếu file kết quả sau bước chạy: {missing}")

if os.path.exists("configs"):
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
elif os.path.exists("self_blancing"):
    os.chdir("self_blancing")
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
else:
    run_step(["git", "clone", "-b", "p1-balance-robot", "https://github.com/nguyenthanhsang2306/self_blancing.git"])
    os.chdir("self_blancing")

run_step([
    sys.executable, "-m", "pip", "install", "-q",
    "gymnasium", "stable-baselines3", "torch", "matplotlib", "pyyaml", "tensorboard"
])
print(f"[Ready] Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"[Check] best_policy_wpos_030.npz có sẵn: {os.path.exists('outputs/checkpoints/best_policy_wpos_030.npz')}")
print(f"[Check] best_policy_wpos_030_sym.npz có sẵn: {os.path.exists('outputs/checkpoints/best_policy_wpos_030_sym.npz')}")

---
### Bước 2: Nạp mốc `best_policy_wpos_030_sym.npz` & Tinh chỉnh ngắn (`50.000` bước, `lr = 3e-5`, chấm mỗi `10.000` bước)

- Nếu runtime Colab chưa có `outputs/checkpoints/best_policy_wpos_030_sym.npz`, ô này tự chuyển từ `best_policy_wpos_030.npz` (nếu có sẵn) hoặc mở hộp thoại để bạn upload `best_policy_wpos_030_sym.npz` / `best_policy_wpos_030.npz`.
- `train.py` nạp đủ trọng số & bias 3 tầng actor vào SB3 PPO, kiểm tra SB3 khớp NumPy/C, chấm lại step 0 xác nhận mốc `57/98` giữ VT và `24/98` bị dừng.
- Cho checkpoint step 0 tham gia xếp hạng; chỉ chọn bản mới nếu giữ VT tốt hơn (`> 57/98`) mà số ca bị dừng không tăng (`<= 24/98`).

In [ ]:
from src.models import NumpyMLPPolicy

ckpt_030_prev = "outputs/checkpoints/best_policy_wpos_030.npz"
ckpt_030_sym = "outputs/checkpoints/best_policy_wpos_030_sym.npz"
os.makedirs("outputs/checkpoints", exist_ok=True)

if not os.path.exists(ckpt_030_sym) and not os.path.exists(ckpt_030_prev):
    print(f"[Notice] Chưa thấy {ckpt_030_sym} hay {ckpt_030_prev} trong Colab runtime.")
    try:
        from google.colab import files
        print("Hãy chọn file best_policy_wpos_030_sym.npz (hoặc best_policy_wpos_030.npz) để upload...")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            base_name = os.path.basename(fname)
            dst = os.path.join("outputs/checkpoints", base_name if base_name.endswith(".npz") else "best_policy_wpos_030_sym.npz")
            with open(dst, "wb") as uf:
                uf.write(data)
            print(f"[Upload] Đã nạp {fname} -> {dst}")
    except Exception as e:
        print(f"[Warning] Không mở được upload ({e}).")

if not os.path.exists(ckpt_030_sym) and os.path.exists(ckpt_030_prev):
    pol_m = NumpyMLPPolicy(input_dim=9, hidden_dims=(64, 64), output_dim=1)
    pol_m.load_weights(ckpt_030_prev)
    pol_m.odd_symmetric = True
    pol_m.save_weights(ckpt_030_sym)
    print(f"[Baseline] Đã tạo bản mốc đối xứng từ {ckpt_030_prev} -> {ckpt_030_sym}")

if not os.path.exists(ckpt_030_sym):
    raise FileNotFoundError(f"Thiếu checkpoint khởi tạo {ckpt_030_sym}. Không thể chạy warm-start PPO!")

run_step(
    [
        sys.executable, "train.py",
        "--config", "configs/default.yaml",
        "--init-weights", ckpt_030_sym,
        "--timesteps", "50000",
        "--learning-rate", "3e-5",
        "--eval-freq", "10000",
        "--w-pos", "0.30",
        "--recenter-prob", "0.0",
        "--top-k", "6",
        "--symmetric",
        "--tag", "wpos_030_5s_sym_nv_ft50k",
    ],
    required_files=[
        "outputs/checkpoints/best_policy_wpos_030_5s_sym_nv_ft50k.npz",
        "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym_nv_ft50k.npz",
        "outputs/checkpoints/final_policy_wpos_030_5s_sym_nv_ft50k.npz",
        "outputs/checkpoints/ppo_final_model_wpos_030_5s_sym_nv_ft50k.zip",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k.h",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k_runner_up.h",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k_final.h",
        "outputs/reports/eval_report_wpos_030_5s_sym_nv_ft50k.md",
    ],
)

---
### Bước 3: Đối chiếu bản mốc step 0 (`best_policy_wpos_030_sym.npz`) với `#1` và `#2` sau tinh chỉnh

- Báo cáo đối chiếu đủ: 98 ca, 2 ca đẩy `0° ±1.5 N`, **30 ca giữ VT**, **20 ca đẩy `7000–7009`** và **20 ca seed mới `7100–7109`** → `outputs/reports/sym_policy_comparison.md`.

In [ ]:
ckpt_030_sym = "outputs/checkpoints/best_policy_wpos_030_sym.npz"
ckpt_sym_best = "outputs/checkpoints/best_policy_wpos_030_5s_sym_nv_ft50k.npz"
ckpt_sym_runner = "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym_nv_ft50k.npz"

missing_ckpts = [p for p in [ckpt_030_sym, ckpt_sym_best, ckpt_sym_runner] if not os.path.exists(p)]
if missing_ckpts:
    raise FileNotFoundError(f"Thiếu checkpoint để so sánh 3 bản: {missing_ckpts}")

run_step(
    [
        sys.executable, "inference.py",
        "--compare",
        ckpt_030_sym,
        ckpt_sym_best,
        ckpt_sym_runner,
        "--export-winner",
        "--report-out", "outputs/reports/sym_policy_comparison.md",
    ],
    required_files=[
        "outputs/checkpoints/ab_winner_policy.npz",
        "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
        "outputs/reports/sym_policy_comparison.md",
    ],
)

---
### Bước 4: Vẽ đồ thị 3 ca còn thiếu của tiêu chí STM32 cho `#1` và `#2`

In [ ]:
from IPython.display import Image, display

plot_jobs = [
    ("nv_ft50k #1 | Góc +3.0° 0 N (10s)", "outputs/checkpoints/best_policy_wpos_030_5s_sym_nv_ft50k.npz", ["--angle", "3.0", "--duration", "10.0"], "outputs/figures/nv_ft50k_best_angle3_10s.png"),
    ("nv_ft50k #1 | Góc -5.0° 0 N (25s)", "outputs/checkpoints/best_policy_wpos_030_5s_sym_nv_ft50k.npz", ["--angle", "-5.0", "--duration", "25.0"], "outputs/figures/nv_ft50k_best_neg5_drift25s.png"),
    ("nv_ft50k #1 | Góc 0.0° -1.5 N (10s)", "outputs/checkpoints/best_policy_wpos_030_5s_sym_nv_ft50k.npz", ["--angle", "0.0", "--kick", "-1.5", "--duration", "10.0"], "outputs/figures/nv_ft50k_best_angle0_kickneg1p5N.png"),
    ("nv_ft50k #2 | Góc +3.0° 0 N (10s)", "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym_nv_ft50k.npz", ["--angle", "3.0", "--duration", "10.0"], "outputs/figures/nv_ft50k_runner_angle3_10s.png"),
    ("nv_ft50k #2 | Góc -5.0° 0 N (25s)", "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym_nv_ft50k.npz", ["--angle", "-5.0", "--duration", "25.0"], "outputs/figures/nv_ft50k_runner_neg5_drift25s.png"),
    ("nv_ft50k #2 | Góc 0.0° -1.5 N (10s)", "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym_nv_ft50k.npz", ["--angle", "0.0", "--kick", "-1.5", "--duration", "10.0"], "outputs/figures/nv_ft50k_runner_angle0_kickneg1p5N.png"),
]

for title, ckpt, extra_args, fig_out in plot_jobs:
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"Dừng vẽ đồ thị vì thiếu checkpoint: {ckpt}")
    print(f"\n=== {title} ===", flush=True)
    run_step(
        [sys.executable, "inference.py", "--weights", ckpt, *extra_args, "--output-figure", fig_out],
        required_files=[fig_out],
    )
    display(Image(filename=fig_out))

---
### Bước 5: Kiểm tra sai số toán học & tính đối xứng xuất C Header (`PyTorch ↔ Numpy ↔ C`)

In [ ]:
run_step([sys.executable, "verify_export.py"])
run_step([
    sys.executable, "verify_export.py",
    "--weights", "outputs/checkpoints/final_policy_wpos_030_5s_sym_nv_ft50k.npz",
    "--sb3-model", "outputs/checkpoints/ppo_final_model_wpos_030_5s_sym_nv_ft50k.zip",
    "--header", "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k_final.h",
])
run_step([
    sys.executable, "verify_export.py",
    "--weights", "outputs/checkpoints/ab_winner_policy.npz",
    "--header", "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
])

---
### Bước 6: Đóng gói `rl_sym_bundle.zip` & Tải về

In [ ]:
import zipfile

required_bundle_files = [
    "outputs/checkpoints/best_policy_wpos_030_sym.npz",
    "outputs/checkpoints/best_policy_wpos_030_5s_sym_nv_ft50k.npz",
    "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym_nv_ft50k.npz",
    "outputs/checkpoints/final_policy_wpos_030_5s_sym_nv_ft50k.npz",
    "outputs/checkpoints/ab_winner_policy.npz",
    "configs/default.yaml",
    "outputs/reports/sym_policy_comparison.md",
    "outputs/reports/eval_report_wpos_030_5s_sym_nv_ft50k.md",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k.h",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k_runner_up.h",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_nv_ft50k_final.h",
    "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
    "outputs/figures/nv_ft50k_best_angle3_10s.png",
    "outputs/figures/nv_ft50k_best_neg5_drift25s.png",
    "outputs/figures/nv_ft50k_best_angle0_kickneg1p5N.png",
    "outputs/figures/nv_ft50k_runner_angle3_10s.png",
    "outputs/figures/nv_ft50k_runner_neg5_drift25s.png",
    "outputs/figures/nv_ft50k_runner_angle0_kickneg1p5N.png",
]

missing_files = [p for p in required_bundle_files if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError(f"Dừng đóng gói vì thiếu file kết quả: {missing_files}")

optional_files = ["outputs/firmware_headers/rl_sb3_stm32.h"]
bundle_files = required_bundle_files + [p for p in optional_files if os.path.exists(p)]

zip_name = "rl_sym_bundle.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in bundle_files:
        zf.write(fpath, arcname=fpath)
        print(f"  + {fpath}")

print(f"\n[Bundle] Đã đóng gói thành công {len(bundle_files)} file vào {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
    files.download("outputs/reports/sym_policy_comparison.md")
except Exception as e:
    print(f"Tự động tải bị chặn ({e}). Hãy tải thủ công từ cột trái Colab: {zip_name}")